# Medicare Part D — Exploratory Data Analysis

This notebook covers:
- Loading data from cache or CMS API
- Schema review and data types
- Statistical summary
- Data dictionary reference
- Overall vs manufacturer record structure
- Outlier flag audit
- GLP-1 drug deep dive

## Imports

In [1]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
import pdfplumber
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display
import skimpy

from src.data_loader import fetch_partd_data, split_overall_mftr, apply_outlier_filter, YEARS
from src.metrics import outlier_summary, fills_per_beneficiary
from src.transforms import build_time_series, top_n_by_metric

## Load Data

In [2]:
df = fetch_partd_data()
df_overall, df_mftr = split_overall_mftr(df)

print(f"Total records:               {len(df):,}")
print(f"Overall (aggregated) rows:   {len(df_overall):,}")
print(f"Manufacturer-level rows:     {len(df_mftr):,}")
print(f"Columns:                     {df.shape[1]}")
print(f"Years covered:               {YEARS}")

Total records:               14,309
Overall (aggregated) rows:   3,598
Manufacturer-level rows:     10,711
Columns:                     47
Years covered:               [2019, 2020, 2021, 2022, 2023]


In [3]:
df.head(10)

,Unnamed: 0,Brnd_Name,Gnrc_Name,Tot_Mftr,Mftr_Name,Tot_Spndng_2019,Tot_Dsg_Unts_2019,Tot_Clms_2019,Tot_Benes_2019,Avg_Spnd_Per_Dsg_Unt_Wghtd_2019,...,Tot_Spndng_2023,Tot_Dsg_Unts_2023,Tot_Clms_2023,Tot_Benes_2023,Avg_Spnd_Per_Dsg_Unt_Wghtd_2023,Avg_Spnd_Per_Clm_2023,Avg_Spnd_Per_Bene_2023,Outlier_Flag_2023,Chg_Avg_Spnd_Per_Dsg_Unt_22_23,CAGR_Avg_Spnd_Per_Dsg_Unt_19_23
0,0,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,139201.68,642471.0,5392.0,1878.0,0.216788,...,44355.04,195672.0,1613,699.0,0.227162,27.498475,63.454993,0.0,0.005702,0.011754
1,1,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Owen Mumford Us,139201.68,642471.0,5392.0,1878.0,0.216788,...,44355.04,195672.0,1613,699.0,0.227162,27.498475,63.454993,0.0,0.005702,0.011754
2,2,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",1,Overall,343031.42,1830596.0,14581.0,5319.0,0.187389,...,97951.18,406617.0,3269,1267.0,0.240932,29.963652,77.309534,0.0,0.022165,0.064848
3,3,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",1,Owen Mumford Us,343031.42,1830596.0,14581.0,5319.0,0.187389,...,97951.18,406617.0,3269,1267.0,0.240932,29.963652,77.309534,0.0,0.022165,0.064848
4,4,Abacavir,Abacavir Sulfate,5,Overall,10110328.45,3316293.0,42629.0,6085.0,3.482725,...,5287295.41,1648593.0,19632,2809.0,3.594357,269.320263,1882.269637,0.0,-0.071481,0.007919
5,5,Abacavir,Abacavir Sulfate,1,"Cipla USA, Inc.",698311.33,192395.0,3311.0,698.0,3.629571,...,710541.39,246306.0,2943,594.0,2.884791,241.434383,1196.197626,0.0,-0.031308,-0.055798
6,6,Abacavir,Abacavir Sulfate,1,Rising Pharm,142257.84,312210.0,433.0,77.0,0.455648,...,48974.57,110779.0,206,31.0,0.442093,237.740631,1579.824839,0.0,-0.058313,-0.007522
7,7,Abacavir,Abacavir Sulfate,1,Mylan,1937920.78,500896.0,7984.0,1505.0,3.868908,...,666703.20,172968.0,2470,469.0,3.854489,269.920324,1421.542004,0.0,-0.123765,-0.000933
8,8,Abacavir,Abacavir Sulfate,1,Aurobindo Pharm,3097563.43,836918.0,13106.0,2299.0,3.701155,...,1411487.34,395492.0,6035,965.0,3.568940,233.883569,1462.681181,0.0,-0.094260,-0.009053
9,9,Abacavir,Abacavir Sulfate,1,Camber Pharmace,4234275.07,1473874.0,17795.0,3305.0,3.194918,...,2449588.91,723048.0,7978,1433.0,3.876191,307.042982,1709.413057,0.0,-0.052507,0.049510


## Schema & Data Types

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14309 entries, 0 to 14308
Data columns (total 47 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Unnamed: 0                       14309 non-null  int64  
 1   Brnd_Name                        14309 non-null  object 
 2   Gnrc_Name                        14309 non-null  object 
 3   Tot_Mftr                         14309 non-null  int64  
 4   Mftr_Name                        14309 non-null  object 
 5   Tot_Spndng_2019                  10104 non-null  float64
 6   Tot_Dsg_Unts_2019                10104 non-null  float64
 7   Tot_Clms_2019                    10104 non-null  float64
 8   Tot_Benes_2019                   10017 non-null  float64
 9   Avg_Spnd_Per_Dsg_Unt_Wghtd_2019  10104 non-null  float64
 10  Avg_Spnd_Per_Clm_2019            10104 non-null  float64
 11  Avg_Spnd_Per_Bene_2019           9910 non-null   float64
 12  Outlier_Flag_2019 

## Statistical Summary

In [5]:
skimpy.skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 14309  │ │ float64     │ 41    │                                                          │
│ │ Number of columns │ 47     │ │ int64       │ 3     │                                                          │
│ └───────────────────┴────────┘ │ string      │ 3     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column  ┃ NA   ┃ NA %    ┃ mean    ┃ sd       ┃ p0      ┃ p25      ┃ p50     ┃ p75      ┃ p100    ┃ hist   ┃  │
│ ┡━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩  │
│ │ Unnamed │    0 │       0 │    7154 │     4131 │       0 │     3577 │    7154 │    10730 │   14310 │ ██████ │  │
│ │ : 0     │      │         │         │          │         │          │         │          │         │        │  │
│ │ Tot_Mft │    0 │       0 │   1.497 │    2.434 │       1 │        1 │       1 │        1 │      45 │   █    │  │
│ │ r       │      │         │         │          │         │          │         │          │         │        │  │
│ │ Tot_Spn │ 4205 │ 29.3870 │ 3542000 │ 20080000 │   57.16 │   114300 │ 1468000 │ 10200000 │ 7305000 │   █    │  │
│ │ dng_201 │      │ 9902858 │       0 │        0 │         │          │         │          │     000 │        │  │
│ │ 9       │      │     341 │         │          │         │          │         │          │         │        │  │
│ │ Tot_Dsg │ 4205 │ 29.3870 │ 2098000 │ 12150000 │     0.7 │    34220 │  362300 │  4186000 │ 3831000 │   █    │  │
│ │ _Unts_2 │      │ 9902858 │       0 │        0 │         │          │         │          │     000 │        │  │
│ │ 019     │      │     341 │         │          │         │          │         │          │         │        │  │
│ │ Tot_Clm │ 4205 │ 29.3870 │  284600 │  1512000 │      11 │      753 │    7167 │    72540 │ 5053000 │   █    │  │
│ │ s_2019  │      │ 9902858 │         │          │         │          │         │          │       0 │        │  │
│ │         │      │     341 │         │          │         │          │         │          │         │        │  │
│ │ Tot_Ben │ 4292 │ 29.9951 │   91380 │   421900 │      11 │      308 │    2473 │    25080 │ 1299000 │   █    │  │
│ │ es_2019 │      │ 0797400 │         │          │         │          │         │          │       0 │        │  │
│ │         │      │    2376 │         │          │         │          │         │          │         │        │  │
│ │ Avg_Spn │ 4205 │ 29.3870 │   176.4 │     1495 │ 0.00023 │   0.4626 │   1.973 │    11.99 │   40190 │   █    │  │
│ │ d_Per_D │      │ 9902858 │         │          │      24 │          │         │          │         │        │  │
│ │ sg_Unt_ │      │     341 │         │          │         │          │         │          │         │        │  │
│ │ Wghtd_2 │      │         │         │          │         │          │         │          │         │        │  │
│ │ 019     │      │         │         │          │         │          │         │          │         │        │  │
│ │ Avg_Spn │ 4205 │ 29.3870 │    1558 │     7147 │ 0.03

## Data Dictionary

In [6]:
dict_path = pathlib.Path("../data/2023/Medicare Part D Spending by Drug Data Dictionary 20250425_508.pdf")

with pdfplumber.open(dict_path) as pdf:
    tables = []
    for page in pdf.pages:
        table = page.extract_table()
        if table:
            tables.extend(table)

dictionary_df = pd.DataFrame(tables[2:], columns=tables[1])
dictionary_df = dictionary_df.dropna()
dictionary_df = dictionary_df[dictionary_df['Variable Name'] != 'Variable Name']

display(dictionary_df.style
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c5f8a'), ('color', 'white'),
                                      ('text-align', 'left'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '6px 12px'), ('border-bottom', '1px solid #ddd')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f0f4f8')]}
    ])
    .hide(axis='index')
)

Variable Name,Term Name,Definition
Brnd_Name,Brand Name,"The name of the drug filled. This includes both brand names (drugs that have a trademarked name) and generic names (drugs that do not have a trademarked name). ' * ' indicates that the average spending per dosage unit reflects multiple routes of administration of the drug (e.g., intravenous, subcutaneous) which individually may have different unit pricing. Additional information regarding calculation of spending per unit can be found in the methodology document."
Gnrc_Name,Generic Name,A term referring to the chemical ingredient of a drug rather than the trademarked brand name under which the drug is sold.
Tot_Mftr,Number of Manufacturers,Number of manufacturers for each drug.
Mftr_Name,Manufacturer,Name of the manufacturer of the drug.
Tot_Spndng_2019,2019 Total Spending,Aggregate drug spending for the Medicare Part D program during the benefit year.
Tot_Dsg_Unts_2019,2019 Total Dosage Units,"The sum of the dosage units of medication dispensed across the calendar year (e.g. number of tablets, grams, milliliters or other units). Unit refers to the drug unit in the lowest dispensable amount."
Tot_Clms_2019,2019 Total Claims,Number of prescription fills for each drug. Includes original prescriptions and refills.
Tot_Benes_2019,2019 Total Beneficiaries,Number of Part D beneficiaries utilizing the drug during the benefit year.
Avg_Spnd_Per_Dsg_Unt_Wghtd_2019,2019 Average Total Spending Per Dosage Unit (Weighted),"Medicare Part D drug spending divided by the number of dosage units, which is weighted by the proportion of total claims."
Avg_Spnd_Per_Clm_2019,2019 Average Total Spending Per Claim,Part D drug spending divided by the number of prescription fills.


## Overall vs Manufacturer Record Structure

In [7]:
drug = "Abacavir"

overall_spend = df_overall[df_overall["Brnd_Name"] == drug]["Tot_Spndng_2023"].values[0]
mftr_sum      = df_mftr[df_mftr["Brnd_Name"] == drug]["Tot_Spndng_2023"].sum()

print(f"Drug:           {drug}")
print(f"Overall spend:  ${overall_spend:,.2f}")
print(f"Sum of mftrs:   ${mftr_sum:,.2f}")
print(f"Match:          {abs(overall_spend - mftr_sum) < 1.0}")

Drug:           Abacavir
Overall spend:  $5,287,295.41
Sum of mftrs:   $5,287,295.41
Match:          True


## Outlier Flag Audit

In [8]:
out_df = outlier_summary(df_overall)
display(out_df.style
    .format({"Outlier_Pct": "{:.2f}%"})
    .bar(subset=["Outlier_Pct"], color="#ff6b6b", vmin=0, vmax=15)
    .hide(axis="index")
)

Year,Total_Records,Outlier_Records,Outlier_Pct
2019,2863,200,6.99%
2020,3055,222,7.27%
2021,3235,252,7.79%
2022,3381,287,8.49%
2023,3585,392,10.93%


## GLP-1 Drug Setup

Define the drug list, verify name matches in the dataset, and check year coverage.

In [10]:
glp1_drugs = ["Ozempic", "Wegovy", "Mounjaro", "Rybelsus",
              "Victoza 2-Pak", "Victoza 3-Pak", "Saxenda", "Trulicity"]

# Verify name matches
print("Name match check:")
for drug in glp1_drugs:
    matches = df_overall[df_overall["Brnd_Name"] == drug]["Brnd_Name"].unique()
    print(f"  {drug!r:35} → {'FOUND' if len(matches) else 'NOT FOUND'}")

# Year coverage
print("\nYear coverage:")
for drug in glp1_drugs:
    row_str = ""
    for yr in YEARS:
        col = f"Tot_Spndng_{yr}"
        val = df_overall[df_overall["Brnd_Name"] == drug][col].values
        has_data = "✓" if len(val) > 0 and pd.notna(val[0]) else "✗"
        row_str += f"{yr}: {has_data}  "
    print(f"  {drug:30} {row_str}")

Name match check:
  'Ozempic'                           → FOUND
  'Wegovy'                            → FOUND
  'Mounjaro'                          → FOUND
  'Rybelsus'                          → FOUND
  'Victoza 2-Pak'                     → FOUND
  'Victoza 3-Pak'                     → FOUND
  'Saxenda'                           → FOUND
  'Trulicity'                         → FOUND

Year coverage:
  Ozempic                        2019: ✓  2020: ✓  2021: ✓  2022: ✓  2023: ✓  
  Wegovy                         2019: ✗  2020: ✗  2021: ✗  2022: ✗  2023: ✓  
  Mounjaro                       2019: ✗  2020: ✗  2021: ✗  2022: ✓  2023: ✓  
  Rybelsus                       2019: ✓  2020: ✓  2021: ✓  2022: ✓  2023: ✓  
  Victoza 2-Pak                  2019: ✓  2020: ✓  2021: ✓  2022: ✓  2023: ✓  
  Victoza 3-Pak                  2019: ✓  2020: ✓  2021: ✓  2022: ✓  2023: ✓  
  Saxenda                        2019: ✗  2020: ✗  2021: ✓  2022: ✗  2023: ✓  
  Trulicity                      2019: ✓  202

## GLP-1 2023 Metrics

In [11]:
glp1_df = df_overall[
    df_overall["Brnd_Name"].isin(glp1_drugs)
][["Brnd_Name", "Gnrc_Name", "Tot_Spndng_2023", "Tot_Clms_2023",
   "Tot_Benes_2023", "Avg_Spnd_Per_Clm_2023", "CAGR_Avg_Spnd_Per_Dsg_Unt_19_23"]].copy()

glp1_df["Fills_Per_Bene"] = fills_per_beneficiary(glp1_df, year=2023).round(1)
glp1_df["CAGR_%"] = (glp1_df["CAGR_Avg_Spnd_Per_Dsg_Unt_19_23"] * 100).round(1)

display(glp1_df.sort_values("Tot_Spndng_2023", ascending=False)
    .style
    .format({
        "Tot_Spndng_2023":       "${:,.0f}",
        "Tot_Clms_2023":         "{:,.0f}",
        "Tot_Benes_2023":        "{:,.0f}",
        "Avg_Spnd_Per_Clm_2023": "${:,.2f}",
        "CAGR_%":                "{:.1f}%",
        "Fills_Per_Bene":        "{:.1f}",
    })
    .hide(axis="index")
)

Brnd_Name,Gnrc_Name,Tot_Spndng_2023,Tot_Clms_2023,Tot_Benes_2023,Avg_Spnd_Per_Clm_2023,CAGR_Avg_Spnd_Per_Dsg_Unt_19_23,Fills_Per_Bene,CAGR_%
Ozempic,Semaglutide,"$9,194,048,435","6,927,972","1,464,468","$1,327.09",-0.041884,4.7,-4.2%
Trulicity,Dulaglutide,"$7,363,856,224","5,316,020","938,731","$1,385.22",0.060886,5.7,6.1%
Mounjaro,Tirzepatide,"$2,361,384,157","1,821,486","370,203","$1,296.41",0.049364,4.9,4.9%
Rybelsus,Semaglutide,"$1,665,906,943","1,075,026","285,693","$1,549.64",0.060467,3.8,6.0%
Victoza 3-Pak,Liraglutide,"$1,004,635,145","574,218","142,213","$1,749.57",0.055057,4.0,5.5%
Victoza 2-Pak,Liraglutide,"$317,213,565","293,462","83,777","$1,080.94",0.050763,3.5,5.1%
Wegovy,Semaglutide,"$199,774",142,47,"$1,406.86",nan,3.0,nan%
Saxenda,Liraglutide,"$33,946",26,nan,"$1,305.61",-0.006246,nan,-0.6%


## GLP-1 Long-Format Data

Build directly from wide columns instead of `build_time_series` so drugs with partial year
coverage (e.g. Wegovy 2023-only, Mounjaro 2022–2023) are included rather than dropped.

In [42]:
import seaborn as sns

glp1_long_rows = []
for drug in glp1_drugs:
    sub = df_overall[df_overall["Brnd_Name"] == drug]
    if sub.empty:
        continue
    for yr in YEARS:
        for metric_key in ["Tot_Spndng", "Tot_Clms", "Tot_Benes"]:
            col = f"{metric_key}_{yr}"
            if col not in sub.columns:
                continue
            val = sub[col].values[0]
            if pd.notna(val):
                glp1_long_rows.append({
                    "Year":   yr,
                    "Drug":   drug,
                    "Metric": metric_key,
                    "Value":  val,
                })

glp1_long_df = pd.DataFrame(glp1_long_rows)

import seaborn as sns

# seaborn "dark" palette, manually mapped
palette = sns.color_palette("deep", 10)
rgb = [f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})" for r, g, b in palette]

color_map = {
    "Ozempic":       rgb[3],
    "Wegovy":        rgb[1],
    "Mounjaro":      rgb[2],
    "Rybelsus":      rgb[0],
    "Victoza 2-Pak": rgb[4],
    "Victoza 3-Pak": rgb[5],
    "Saxenda":       rgb[6],
    "Trulicity":     rgb[9],
}

drug_list = [d for d in glp1_drugs if d in glp1_long_df["Drug"].values]


## GLP-1 Stacked Bar Charts · Total Spending, Claims, Beneficiaries

In [43]:
plot_configs = [
    ("Tot_Spndng", "Total Spending ($)",   lambda v: f"${v/1e6:,.1f}M"),
    ("Tot_Clms",   "Total Claims",          lambda v: f"{v:,.0f}"),
    ("Tot_Benes",  "Total Beneficiaries",   lambda v: f"{v:,.0f}"),
]

for metric_key, metric_label, fmt in plot_configs:
    subset = glp1_long_df[glp1_long_df["Metric"] == metric_key]

    fig = go.Figure()
    for drug in drug_list:
        drug_data = subset[subset["Drug"] == drug].sort_values("Year")
        fig.add_trace(go.Bar(
            name=drug,
            x=drug_data["Year"],
            y=drug_data["Value"],
            marker_color=color_map[drug],
            hovertemplate=(
                f"<b>{drug}</b><br>"
                "Year: %{x}<br>"
                f"{metric_label}: %{{customdata}}<extra></extra>"
            ),
            customdata=[fmt(v) for v in drug_data["Value"]],
        ))

    fig.update_layout(
        barmode="stack",
        title=dict(
            text=f"GLP-1 Drugs · {metric_label} · 2019–2023",
            font=dict(family="DM Serif Display, serif", size=22, color="#c8d4f0"),
            x=0.02,
        ),
        height=420,
        paper_bgcolor="#0f1117",
        plot_bgcolor="#141824",
        font=dict(family="DM Sans, sans-serif", color="#c8d4f0"),
        legend=dict(
            title=dict(text="Drug", font=dict(size=12, color="#7b8bb2")),
            orientation="v", x=1.01, y=1,
            bgcolor="rgba(0,0,0,0)", bordercolor="#2a3a5c", borderwidth=1,
            font=dict(size=12),
        ),
        hovermode="closest",
        margin=dict(l=60, r=160, t=60, b=50),
        xaxis=dict(tickvals=YEARS, tickformat="d", showgrid=False,
                               linecolor="#2a3a5c", tickcolor="#2a3a5c"),
        yaxis=dict(gridcolor="#1e2a40", linecolor="#2a3a5c", tickcolor="#2a3a5c",
                   tickprefix="$" if metric_key == "Tot_Spndng" else ""),
        bargap=0.25,
    )
    fig.show()

## GLP-1 Line Chart · Avg Spending per Dosage Unit

In [44]:
glp1_ts_rows = []
for drug in glp1_drugs:
    sub = df_overall[df_overall["Brnd_Name"] == drug]
    if sub.empty:
        continue
    for yr in YEARS:
        col = f"Avg_Spnd_Per_Dsg_Unt_Wghtd_{yr}"
        if col not in sub.columns:
            continue
        val = sub[col].values[0]
        if pd.notna(val):
            glp1_ts_rows.append({"Year": yr, "Drug": drug, "Avg_Cost_Per_Unit": val})

glp1_ts_df = pd.DataFrame(glp1_ts_rows)

fig = go.Figure()
for drug in drug_list:
    drug_data = glp1_ts_df[glp1_ts_df["Drug"] == drug].sort_values("Year")
    if drug_data.empty:
        continue
    fig.add_trace(go.Scatter(
        name=drug,
        x=drug_data["Year"],
        y=drug_data["Avg_Cost_Per_Unit"],
        mode="lines+markers",
        line=dict(width=2.5, color=color_map[drug]),
        marker=dict(size=8, color=color_map[drug]),
        hovertemplate=(
            f"<b>{drug}</b><br>"
            "Year: %{x}<br>"
            "Avg Spend per Dosage Unit: $%{y:,.2f}<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(
        text="GLP-1 Drugs · Avg Spending per Dosage Unit · 2019–2023",
        font=dict(family="DM Serif Display, serif", size=22, color="#c8d4f0"),
        x=0.02,
    ),
    height=420,
    paper_bgcolor="#0f1117",
    plot_bgcolor="#141824",
    font=dict(family="DM Sans, sans-serif", color="#c8d4f0"),
    legend=dict(
        title=dict(text="Drug", font=dict(size=12, color="#7b8bb2")),
        orientation="v", x=1.01, y=1,
        bgcolor="rgba(0,0,0,0)", bordercolor="#2a3a5c", borderwidth=1,
        font=dict(size=12),
    ),
    hovermode="closest",
    margin=dict(l=60, r=160, t=60, b=50),
    xaxis=dict(tickvals=YEARS, tickformat="d", gridcolor="#1e2a40",
               linecolor="#2a3a5c", tickcolor="#2a3a5c"),
    yaxis=dict(gridcolor="#1e2a40", linecolor="#2a3a5c", tickcolor="#2a3a5c",
               tickprefix="$"),
)
fig.show()

## GLP-1 Line Chart · Total Spending Trend

In [45]:
glp1_spend_rows = []
for drug in glp1_drugs:
    sub = df_overall[df_overall["Brnd_Name"] == drug]
    if sub.empty:
        continue
    for yr in YEARS:
        col = f"Tot_Spndng_{yr}"
        if col not in sub.columns:
            continue
        val = sub[col].values[0]
        if pd.notna(val):
            glp1_spend_rows.append({"Year": yr, "Drug": drug, "Total_Spending": val})

glp1_spend_df = pd.DataFrame(glp1_spend_rows)

fig = go.Figure()
for drug in drug_list:
    drug_data = glp1_spend_df[glp1_spend_df["Drug"] == drug].sort_values("Year")
    if drug_data.empty:
        continue
    fig.add_trace(go.Scatter(
        name=drug,
        x=drug_data["Year"],
        y=drug_data["Total_Spending"],
        mode="lines+markers",
        line=dict(width=2.5, color=color_map[drug]),
        marker=dict(size=8, color=color_map[drug]),
        hovertemplate=(
            f"<b>{drug}</b><br>"
            "Year: %{x}<br>"
            "Total Spending: $%{y:,.0f}<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(
        text="GLP-1 Drugs · Total Spending Trend · 2019–2023",
        font=dict(family="DM Serif Display, serif", size=22, color="#c8d4f0"),
        x=0.02,
    ),
    height=420,
    paper_bgcolor="#0f1117",
    plot_bgcolor="#141824",
    font=dict(family="DM Sans, sans-serif", color="#c8d4f0"),
    legend=dict(
        title=dict(text="Drug", font=dict(size=12, color="#7b8bb2")),
        orientation="v", x=1.01, y=1,
        bgcolor="rgba(0,0,0,0)", bordercolor="#2a3a5c", borderwidth=1,
        font=dict(size=12),
    ),
    hovermode="closest",
    margin=dict(l=60, r=160, t=60, b=50),
    xaxis=dict(tickvals=YEARS, tickformat="d", gridcolor="#1e2a40",
               linecolor="#2a3a5c", tickcolor="#2a3a5c"),
    yaxis=dict(gridcolor="#1e2a40", linecolor="#2a3a5c", tickcolor="#2a3a5c",
               tickprefix="$"),
)
fig.show()

## Top 20 Drugs by 2023 Spend

In [48]:
top20 = top_n_by_metric(df_overall, metric_col="Tot_Spndng_2023", n=20,
                        exclude_outliers=True, year=2023)
display(top20.style
    .format({"Tot_Spndng_2023": "${:,.0f}"})
    .bar(subset=["Tot_Spndng_2023"], color="#4e9af1")
    .hide(axis="index")
)

Brnd_Name,Gnrc_Name,Mftr_Name,Tot_Spndng_2023
Eliquis,Apixaban,Overall,"$18,273,451,967"
Ozempic,Semaglutide,Overall,"$9,194,048,435"
Jardiance,Empagliflozin,Overall,"$8,839,935,063"
Trulicity,Dulaglutide,Overall,"$7,363,856,224"
Xarelto,Rivaroxaban,Overall,"$6,309,246,823"
Trelegy Ellipta,Fluticasone/Umeclidin/Vilanter,Overall,"$4,455,884,010"
Humira(Cf) Pen,Adalimumab,Overall,"$4,419,828,188"
Farxiga,Dapagliflozin Propanediol,Overall,"$4,342,182,307"
Januvia,Sitagliptin Phosphate,Overall,"$4,090,836,821"
Revlimid,Lenalidomide,Overall,"$3,859,804,789"
